# Scrye — Analysis Notebook (default = zero-shot baseline)

Runs the full prediction → scoring → analysis chain on the **default zero-shot pipeline** and renders the visualization suite. Re-run top to bottom any time: LLM calls are cached on disk, so a repeat run is fast, free, and deterministic.

## The swappable chain

```
SimBenchRecord ──▶ Predictor ──▶ Calibrator ──▶ prediction ──▶ evaluate() ──▶ viz
                  (simulation)   (correction)
```

Everything below the **Configure the pipeline** cell is agnostic to what's inside the pipeline. To try a new approach later, implement `scrye.predict.Predictor` or `scrye.calibrate.Calibrator` and swap it in that one cell — the entire analysis re-runs unchanged.

In [ ]:
%load_ext autoreload
%autoreload 2

from scrye.config import DEFAULT_MODEL, FIGURES_DIR, OUTPUTS_DIR
from scrye.data import load_split
from scrye.llm import LLMClient
from scrye.predict import ZeroShotPredictor, UniformPredictor
from scrye.calibrate import IdentityCalibrator
from scrye.pipeline import Pipeline
from scrye import evaluate as ev
from scrye import viz

pop = load_split("pop")
grouped = load_split("grouped")

# Per-dataset SimBench Eq. 2 normalizers, built from the FULL data so scores are
# stable and comparable to the paper regardless of the eval subsample.
normalizers = ev.build_normalizers(pop + grouped)
print(f"loaded: pop={len(pop)}  grouped={len(grouped)}  | default model: {DEFAULT_MODEL}")
print(f"normalizers for {len(normalizers)} (split, dataset) groups")

## Configure the pipeline  ← swap components here

The default is `ZeroShotPredictor` + `IdentityCalibrator` (no calibration). Replace either stage with a new implementation of the same interface to test an approach.

In [ ]:
client = LLMClient(model=DEFAULT_MODEL, max_tokens=256, temperature=0.0)

predictor = ZeroShotPredictor(client)     # <-- the 'simulation' component
calibrator = IdentityCalibrator()         # <-- the 'calibration' component (no-op default)
pipeline = Pipeline(predictor, calibrator)

print("pipeline:", pipeline.name)

## Build the evaluation sample

A seeded, dataset-stratified sample for the headline analysis, plus all variants of the three assignment-required questions. Small enough to be cheap; the cache makes re-runs instant. Bump `N` for tighter confidence intervals.

In [ ]:
N = 300
sample = ev.stratified_sample(pop + grouped, n=N, seed=0)
print(f"evaluation sample: {len(sample)} records across "
      f"{len({r.dataset_name for r in sample})} datasets")

required = ev.required_question_records(grouped)
for k, recs in required.items():
    print(f"  required[{k}]: {len(recs)} segment variants")

## Run the pipeline and summarize

In [ ]:
results = ev.evaluate(pipeline, sample, normalizers=normalizers, max_workers=8)

summary = ev.summarize(results)
print(f"pipeline           : {summary['pipeline']}")
print(f"n                  : {summary['n']}")
print(f"mean SimBench S    : {summary['mean_score']:.1f}  "
      f"(95% CI {summary['ci_low']:.1f}, {summary['ci_high']:.1f})")
print(f"frac below uniform : {summary['frac_below_uniform']:.1%}")
print(f"parse failures     : {predictor.n_parse_failures}")
print(f"LLM usage          : {client.usage.as_dict()}")

In [ ]:
ev.score_by(results, "dataset")

## Visualization suite

Each figure diagnoses the predictor, not just scores it. All take the results DataFrame, so they work for any pipeline.

In [ ]:
viz.plot_score_histogram(results)

In [ ]:
viz.plot_score_by_dataset(results)

In [ ]:
# The mode-seeking test: does the model fail on high-entropy (divided) questions?
viz.plot_score_vs_entropy(results)

In [ ]:
viz.plot_score_by_n_options(results)

In [ ]:
# Deviation from the diagonal is what a calibration component would correct.
viz.plot_calibration(results)

In [ ]:
viz.plot_example_distributions(results, n=6)

## Counterfactual sensitivity

Does conditioning on a segment move the prediction in the *right direction* (vs. the population average), independent of unconditioned-prior accuracy? Mean cosine alignment of predicted vs. true shifts over all segment/question pairs. Run on the grouped split, which has the segment variants.

In [ ]:
# Evaluate on grouped segments of the required questions (rich in variants).
cf_records = [r for recs in required.values() for r in recs]
cf_results = ev.evaluate(pipeline, cf_records, normalizers=normalizers, max_workers=8)
mean_align, cf_table = ev.counterfactual_sensitivity(cf_results)
print(f"mean counterfactual alignment: {mean_align:+.3f}  "
      f"(+1 right direction, -1 wrong, 0 none) over {len(cf_table)} segments")
cf_table.head(10)

## Required-question deep dive

In [ ]:
import pandas as pd
rows = []
for key, recs in required.items():
    sub = cf_results[cf_results["input_template"].isin({r.input_template for r in recs})]
    if len(sub):
        rows.append({"question": key, "n_variants": len(sub),
                     "mean_S": sub["score"].mean()})
pd.DataFrame(rows)

## Save headline figures for the writeup

Presentation-final figures go to `docs/figures/` (tracked); scratch stays in `outputs/` (gitignored).

In [ ]:
tag = pipeline.name.replace("+", "_").replace("/", "-")
viz.plot_score_histogram(results).savefig(FIGURES_DIR / f"score_hist__{tag}.png", dpi=120)
viz.plot_score_by_dataset(results).savefig(FIGURES_DIR / f"score_by_dataset__{tag}.png", dpi=120)
viz.plot_score_vs_entropy(results).savefig(FIGURES_DIR / f"score_vs_entropy__{tag}.png", dpi=120)
viz.plot_calibration(results).savefig(FIGURES_DIR / f"calibration__{tag}.png", dpi=120)
print("saved figures to", FIGURES_DIR)